# Cours — Réseaux de neurones convolutifs (CNN)

**Semaine 5 · Jour 3** · Propilote-Learning

**Au programme**

1. Introduction aux réseaux de neurones convolutifs (CNN)
2. Applications des CNN et implémentation pratique
3. Tutoriel : classification d'images avec un CNN (chat vs. chien)

> **Comment utiliser ce notebook**
>
> 1. **Fichier → Enregistrer une copie dans Drive** pour travailler sur ta propre copie.
> 2. Lis chaque leçon et **exécute les cellules de code** (Maj + Entrée) : modifie-les, teste, observe.
> 3. Les lignes `# → …` indiquent le résultat attendu.
> 4. Les cellules **✍️ Essaie ici** sont pour toi : fais les mini-exercices du cours.
> 5. Certaines cellules montrent **volontairement une erreur** (le texte autour l'explique) : lis le message, c'est le but.
> 6. Les exercices notés de la journée sont dans les notebooks d'exercices (liens en bas).

# 1. Introduction aux réseaux de neurones convolutifs (CNN)

*Pourquoi les CNN surpassent les ANN classiques sur les images, et comment ils fonctionnent*

**Objectifs**

- Comprendre les concepts de base et l'architecture des CNN
- Comprendre en quoi les CNN diffèrent des réseaux de neurones artificiels (ANN) classiques
- Explorer les composants clés des CNN : couches convolutives, de pooling et entièrement connectées
- Comprendre comment les CNN traitent et interprètent des données visuelles

## Qu'est-ce qu'un CNN ?

Imagine un réseau de neurones convolutif (CNN) comme un type particulier de réseau de neurones (ANN) conçu pour voir et comprendre des images, un peu comme notre cerveau traite l'information visuelle.

#### En quoi un CNN diffère-t-il d'un ANN classique ?

Dans un ANN classique, chaque neurone est connecté à tous les neurones de la couche suivante (fully connected). Cela fonctionne bien pour des données structurées (comme des nombres dans un tableur), mais les images ont une structure spatiale — les pixels voisins sont liés entre eux !

## Pourquoi ne pas simplement utiliser des ANN pour les images ?

### A. Trop de connexions

Prenons un exemple : une petite image en niveaux de gris de 100×100 pixels compte 10 000 pixels au total. En aplatissant l'image, il faut 10 000 neurones d'entrée — un par pixel. Si la première couche cachée compte 1 000 neurones entièrement connectés, cela donne 10 000 × 1 000 = 10 millions de connexions, rien que pour une couche !

Pour une image haute résolution de 1080×1920 pixels (Full HD), soit plus de 2 millions de pixels, on obtiendrait des milliards de connexions. Le réseau deviendrait très lent à entraîner, extrêmement gourmand en mémoire, et sujet à l'overfitting à cause du trop grand nombre de paramètres.

Les CNN évitent cette explosion de paramètres en utilisant des filtres locaux (aussi appelés kernels) qui balaient l'image. Plutôt que de connecter chaque pixel à chaque neurone, les CNN connectent de petites régions de l'image (des patchs de 3×3 ou 5×5) à des neurones individuels — ce qui réduit drastiquement le nombre de paramètres tout en capturant les features spatiales locales (contours, courbes, textures).

### B. Perte d'information spatiale

Les ANN traitent chaque pixel comme une entrée indépendante : ils ne tiennent pas compte de la position ou de la relation entre pixels voisins, et échouent donc à capturer la structure spatiale d'une image — alors que dans une image réelle, les relations spatiales comptent (les yeux d'un chat sont proches de son nez, par exemple).

Les CNN résolvent ce problème grâce à des couches convolutives qui préservent les hiérarchies spatiales : elles analysent de petites sections de l'image (3×3 ou 5×5) et glissent sur toute l'image, ce qui leur permet de détecter des features locales et de les combiner en motifs de plus haut niveau.

> 💡 **À retenir** : En résumé : les CNN comprennent où se trouvent les choses dans une image — les ANN, non.

### C. Mauvaise généralisation à de nouvelles images

Les ANN ont tendance à mémoriser les motifs des données d'entraînement plutôt qu'à apprendre leur structure sous-jacente — d'où de mauvaises performances sur des images légèrement modifiées (tournées, zoomées, floues ou décalées).

Les CNN, eux, sont conçus pour être invariants par translation : ils scannent l'image avec des filtres qui détectent des features (contours, textures) quel que soit l'endroit où elles apparaissent, ce qui les rend plus robustes aux changements de position, d'échelle ou de netteté.

## Comment fonctionnent les CNN

Contrairement aux réseaux traditionnels qui traitent toutes les valeurs de pixels d'un coup, les CNN décomposent l'image en sections plus petites et les traitent à travers plusieurs couches. À chaque couche, le réseau apprend à détecter des motifs de plus en plus complexes.

#### Étape 1 : la convolution — détecter les features

Dans la première couche, le CNN fait glisser de petites « fenêtres » — appelées filtres ou kernels — sur l'image. Ces filtres détectent des features basiques comme des contours, des lignes, des textures simples. Chaque fois que le filtre glisse sur une partie de l'image, il produit un nombre représentant l'intensité de cette feature dans cette région. Ce processus s'appelle la convolution.

```python
7 2 3 3 8      1  0 -1
4 5 3 8 4      1  0 -1     =>  6 ...
3 3 2 8 4  *   1  0 -1
2 8 7 2 7
5 4 4 5 4
```

In [ ]:
# 7*1 + 4*1 + 3*1 + 2*0 + 5*0 + 3*0 + 3*-1 + 3*-1 + 2*-1 = 6

Sur plusieurs couches, le CNN combine ces détections locales pour identifier des features plus abstraites : les premières couches détectent des contours, les suivantes combinent ces contours en formes, et les couches profondes reconnaissent des objets (yeux, roues, feuilles…).

#### Étape 2 : le pooling — simplifier sans perdre l'essentiel

Une fois les features détectées, l'étape suivante consiste à simplifier les données tout en conservant l'information la plus importante — le pooling. C'est comme zoomer en arrière sur une photo : on perd des détails fins, mais la structure globale reste visible.

La méthode la plus courante est le max pooling :

- La feature map est découpée en petites régions (généralement 2×2 ou 3×3).
- Pour chaque région, seule la valeur maximale est conservée — le signal le plus fort.
- Toutes les autres valeurs sont ignorées.

Cela réduit la quantité de données à traiter (modèle plus rapide et efficace) et ajoute une certaine flexibilité spatiale, en tolérant de légers déplacements d'une feature dans l'image.

#### Étape 3 : convolutions supplémentaires — affiner la compréhension

Après le pooling, le CNN continue d'appliquer des couches convolutives supplémentaires (souvent avec des filtres plus petits, comme 2×2) pour affiner et approfondir sa compréhension de l'image. Le réseau ne se contente plus de détecter des contours : il combine ces features précoces pour identifier des motifs plus complexes et abstraits (un œil, une feuille, une roue).

#### Étape 4 : la couche entièrement connectée — la décision finale

Après convolution et pooling, les feature maps structurées doivent être aplaties (flatten) en un vecteur unidimensionnel. Ce vecteur passe ensuite dans une ou plusieurs couches entièrement connectées (comme dans un ANN classique), qui apprennent à associer les motifs détectés à des catégories spécifiques.

**Sortie :**

```text
0.01 (chien), 0.02 (lapin), 0.96 (chat), 0.01 (cheval)
# Le modèle classe l'image avec confiance comme un chat
```

#### Étape 5 : la carte de probabilité (sortie finale)

Dans certaines applications (détection d'objets, imagerie médicale), la sortie finale n'est pas juste une classification unique, mais une carte de probabilité (heatmap) qui montre visuellement où, dans l'image, certaines features ou certains objets sont le plus probablement situés — les zones chaudes (rouge/jaune) indiquant une forte probabilité, les zones froides (bleu/noir) une probabilité faible.

> ⚠️ **Important** : Ce type de sortie est particulièrement utile pour interpréter les décisions du modèle (explicabilité), localiser des objets, tumeurs ou défauts, et renforcer la confiance dans les prédictions de l'IA dans des domaines sensibles comme la santé ou les véhicules autonomes.

> ✏️ **À toi**
>
> Un CNN a été entraîné à reconnaître des chiffres manuscrits (0-9).
>
> - Pourquoi les CNN utilisent-ils des couches de pooling ? Que se passerait-il si on les supprimait ?
> - Un filtre 3×3 a les valeurs suivantes : [1 0 -1 / 1 0 -1 / 1 0 -1]. Quel type de feature ce filtre détecte-t-il selon toi ?

In [ ]:
# ✍️ Essaie ici


## Les composants clés d'un CNN

L'architecture couramment utilisée pour des tâches comme la reconnaissance de chiffres manuscrits (dataset MNIST) illustre bien ces composants : elle traite les images étape par étape, transformant les valeurs de pixels brutes en prédictions.

#### 1. Les couches convolutives — extraire les premiers indices

- L'image est analysée avec de petits filtres apprenables, appelés kernels (souvent 5×5 ou 3×3).
- Ces filtres glissent sur l'image (convolution), en se concentrant sur de petites régions à la fois.
- À chaque position, le filtre effectue un produit scalaire entre ses poids et les valeurs de pixels de la région couverte.
- Le résultat est une feature map qui met en évidence où certains motifs (un contour vertical, une courbe…) sont présents.

#### 2. Les couches de pooling — simplifier en conservant l'essentiel

Le pooling (ou downsampling) a un double objectif : réduire les dimensions spatiales des feature maps, et préserver les features clés pour que le réseau soit plus efficace et moins sujet à l'overfitting.

- Max pooling : sélectionne la valeur maximale de chaque région (2×2 ou 3×3 en général) — la méthode la plus utilisée.
- Average pooling : prend la valeur moyenne de chaque région, utile quand l'intensité globale compte plus que les pics.

Exemple sur MNIST : une première couche de max pooling réduit la taille de 24×24 à 12×12 ; une seconde la réduit encore à 4×4, ne conservant que les features les plus importantes.

#### 3. Les couches entièrement connectées — la prédiction finale

Les feature maps de la dernière couche de pooling sont d'abord aplaties en un vecteur unidimensionnel. Ce vecteur passe ensuite dans une ou plusieurs couches entièrement connectées (dense), où chaque neurone est connecté à tous les neurones de la couche précédente.

- L'activation ReLU introduit de la non-linéarité, essentielle pour modéliser des relations complexes.
- Une couche de dropout réduit le risque d'overfitting en désactivant aléatoirement certains neurones pendant l'entraînement.
- La couche de sortie utilise une activation softmax qui convertit les scores bruts en probabilités sommant à 1 — le chiffre avec la probabilité la plus élevée est choisi comme prédiction.

| Étape | Rôle |
| --- | --- |
| Couche d'entrée | Image 28×28 en niveaux de gris d'un chiffre |
| Convolution 1 (5×5) | Extrait des features de bas niveau (contours, courbes) |
| Max pooling 1 (2×2) | Réduit la taille de la feature map à 12×12 |
| Convolution 2 (5×5) | Extrait des motifs plus complexes |
| Max pooling 2 (2×2) | Réduit la taille à 4×4 |
| Flatten | Convertit les feature maps en vecteur 1D |
| Couches entièrement connectées | Traitent les features avec activation ReLU |
| Couche de sortie (softmax) | Produit une probabilité pour chaque chiffre (0-9) |

> ⚠️ **Important** : À retenir : les couches convolutives apprennent des motifs (contours, formes) ; les couches de pooling réduisent la dimensionnalité et le calcul ; les couches entièrement connectées effectuent la classification finale ; le dropout prévient l'overfitting et softmax convertit les sorties en probabilités.

> ✏️ **À toi**
>
> Pourquoi les CNN utilisent-ils des couches entièrement connectées à la fin plutôt que de continuer avec davantage de couches convolutives ? Pourrait-on remplacer ces couches par du global pooling ? Quel serait l'avantage ou l'inconvénient ?

In [ ]:
# ✍️ Essaie ici


# 2. Applications des CNN et implémentation pratique

*Reconnaissance d'images, détection d'objets, imagerie médicale · exemple CIFAR-10 en TensorFlow et PyTorch*

**Objectifs**

- Comprendre les principales applications des CNN dans des scénarios réels
- Apprendre à prétraiter des données, définir une architecture CNN, et entraîner un modèle avec TensorFlow et Keras
- Découvrir comment améliorer la performance d'un CNN via data augmentation et réglage d'hyperparamètres
- Comparer TensorFlow/Keras et PyTorch

## Applications des CNN

### A. Reconnaissance et classification d'images

Les CNN peuvent classer des images dans des catégories prédéfinies en apprenant des motifs, textures et formes. Le modèle est entraîné sur un dataset labellisé (comme ImageNet, des millions d'images réparties en milliers de catégories) et apprend à reconnaître les features clés qui définissent chaque objet.

- Google Lens et Apple Photos utilisent des CNN pour classer les images en catégories comme « montagnes », « nourriture » ou « animaux ».
- Les véhicules autonomes utilisent une classification basée sur les CNN pour différencier panneaux, piétons et véhicules.

### B. Détection d'objets

Contrairement à la classification (qui assigne une seule catégorie à une image), la détection d'objets identifie plusieurs objets dans une image et localise leur position, via des architectures comme YOLO (You Only Look Once), Faster R-CNN ou SSD (Single Shot Detector).

- Les voitures autonomes détectent piétons, feux de circulation et marquages au sol.
- Les systèmes de vidéosurveillance détectent des intrus ou des activités suspectes.
- Le e-commerce utilise la détection d'objets pour des expériences de shopping en réalité augmentée.

### C. Reconnaissance faciale

Les CNN analysent des features faciales (distance entre les yeux, forme du nez, contours de la mâchoire) et convertissent les images de visages en vecteurs numériques comparés à des embeddings stockés. Des techniques avancées comme les Siamese Networks ou FaceNet améliorent la précision en rapprochant les visages similaires dans un espace de haute dimension.

### D. Analyse d'images médicales

Les CNN aident au diagnostic de maladies en analysant des images médicales (radios, scanners, IRM, lames histopathologiques), en apprenant à reconnaître des motifs anormaux comme des tumeurs, fractures ou infections — par exemple avec des modèles comme CheXNet pour détecter des pneumonies sur des radios pulmonaires.

## Exemple pratique avec TensorFlow/Keras

Prenons un exemple concret d'utilisation d'un CNN pour la classification d'images, en entraînant un CNN sur le dataset CIFAR-10 (60 000 images réparties en 10 classes : avions, voitures, oiseaux, chats, cerfs, chiens, grenouilles, chevaux, bateaux, camions).

#### Étape 1 : importer les librairies

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator

#### Étape 2 : charger et prétraiter les données

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normaliser les images (valeurs de pixels entre 0 et 1)
x_train, x_test = x_train / 255.0, x_test / 255.0

#### Étape 3 : data augmentation

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow(x_train, y_train, batch_size=32)
test_generator = test_datagen.flow(x_test, y_test, batch_size=32)

#### Étape 4 : définir le modèle CNN

In [ ]:
model = Sequential()

# Première couche convolutive
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Deuxième couche convolutive
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Aplatissement
model.add(Flatten())

# Couches entièrement connectées
model.add(Dense(units=128, activation='relu'))
model.add(Dense(units=10, activation='softmax'))

#### Étape 5 : compiler et entraîner

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_generator, validation_data=test_generator, epochs=10)

#### Étape 6 : évaluer

In [ ]:
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {test_accuracy:.4f}")

**Sortie :**

```text
Epoch 10/10
1563/1563 - accuracy: 0.4054 - loss: 1.6773 - val_accuracy: 0.4060
Test Accuracy: 0.4060
```

> ⚠️ **Important** : Une accuracy de 40,6 % après seulement 10 epochs montre que le modèle a appris, mais nécessite encore du réglage. Ajouter des couches convolutives, augmenter le nombre de filtres, utiliser la batch normalization et augmenter le nombre d'epochs (à 50 par exemple) peuvent tous améliorer la performance.

> ✏️ **À toi**
>
> Ton CNN entraîné sur CIFAR-10 a atteint 40,6 % d'accuracy après 10 epochs. Tu décides d'ajouter des couches convolutives, d'augmenter le nombre de filtres par couche, d'utiliser la batch normalization après chaque couche convolutive, et de passer à 50 epochs. Comment chacun de ces changements affecterait-il l'apprentissage et la performance finale du modèle ?

In [ ]:
# ✍️ Essaie ici


## Le même exemple avec PyTorch

PyTorch est l'un des frameworks de deep learning les plus puissants et utilisés aujourd'hui, connu pour sa flexibilité et son design intuitif. Développé par Facebook AI (Meta), il utilise des graphes de calcul dynamiques : on peut modifier la structure d'un modèle à la volée pendant l'exécution, ce qui facilite grandement le débogage et l'expérimentation.

PyTorch fonctionne avec des tenseurs, similaires aux tableaux NumPy mais avec accélération GPU. Il dispose d'un moteur de différenciation automatique (autograd) qui calcule automatiquement les gradients.

In [ ]:
import torch
import numpy as np

# Créer un tenseur depuis une liste Python
tensor_list = torch.tensor([1, 2, 3, 4, 5])

# Créer un tenseur depuis un tableau NumPy
numpy_array = np.array([[1, 2], [3, 4]])
tensor_numpy = torch.tensor(numpy_array)

# Opérations sur les tenseurs
tensor_a = torch.tensor([[1, 2], [3, 4]])
tensor_b = torch.tensor([[5, 6], [7, 8]])
tensor_sum = tensor_a + tensor_b
tensor_matmul = torch.matmul(tensor_a, tensor_b)

#### Définir le modèle CNN en PyTorch

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)   # 3 canaux d'entrée (RGB), 32 filtres, noyau 3x3
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 32, 3)
        self.fc1 = nn.Linear(32 * 6 * 6, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters())

On entraîne ensuite le modèle sur plusieurs epochs en itérant sur les batches, en remettant les gradients à zéro (zero_grad), en calculant la sortie et la perte, puis en rétropropageant l'erreur (loss.backward()) et en mettant à jour les poids (optimizer.step()).

## TensorFlow/Keras ou PyTorch ?

| Critère | TensorFlow/Keras | PyTorch |
| --- | --- | --- |
| Niveau d'API | Haut niveau (Keras) : simple et rapide | Plus bas niveau, flexible |
| Graphe de calcul | Historiquement statique (TF 2.x plus dynamique) | Dynamique — modifiable en cours d'exécution |
| Débogage | Plus difficile avec les graphes statiques | Plus facile grâce au graphe dynamique |
| Production | Écosystème très mature (TF Serving, TF Lite) | S'améliore (TorchServe) mais moins mature |
| Recherche | Moins privilégié en recherche académique | Très populaire en recherche |
| Idéal pour | Débutants, déploiement à grande échelle | Contrôle fin, prototypage, recherche |

Le meilleur choix dépend du projet : TensorFlow/Keras pour démarrer rapidement et déployer en production à grande échelle, PyTorch pour la recherche, la flexibilité et le contrôle fin des modèles.

> ✏️ **À toi**
>
> Pourquoi les CNN surpassent-ils les algorithmes de machine learning traditionnels (comme les SVM ou les arbres de décision) sur les tâches liées aux images ?

In [ ]:
# ✍️ Essaie ici


# 3. Tutoriel : classification d'images avec un CNN (chat vs. chien)

*Prétraitement d'images, architecture TFLearn/TensorFlow, entraînement et test*

**Objectifs**

- Comprendre les bases des réseaux de neurones convolutifs (CNN)
- Apprendre à prétraiter des données d'images pour le machine learning
- Construire, entraîner et évaluer un modèle CNN avec TFLearn et TensorFlow
- Visualiser les résultats de son classifieur d'images

## Le problème et la mise en place de l'environnement

Dans ce tutoriel, on construit un classifieur d'images capable de distinguer les chats des chiens, en utilisant Python, TensorFlow et TFLearn — une API haut niveau qui simplifie la construction et l'entraînement de modèles de deep learning.

#### Installer les librairies nécessaires

> ⚠️ **À lire** : ce code utilise tflearn, une bibliothèque qui n'est plus maintenue et ne fonctionne pas avec TensorFlow 2 (la version de Colab). Lis-le pour comprendre l'architecture ; pour l'exécuter, réécris le modèle avec Keras (tf.keras.Sequential, Conv2D, MaxPooling2D, Dense), comme dans les autres exemples de la semaine.

```python
!pip install tensorflow tflearn opencv-python numpy tqdm matplotlib
```

Le dataset (Dogs vs. Cats) doit être organisé en deux dossiers : train (images de chats et de chiens labellisées) et test1 (images non labellisées pour le test).

#### Importer les librairies

```python
import cv2
import os
import numpy as np
from random import shuffle
from tqdm import tqdm
import tflearn
from tflearn.layers.conv import conv_2d, max_pool_2d
from tflearn.layers.core import input_data, dropout, fully_connected
from tflearn.layers.estimator import regression
import tensorflow as tf
import matplotlib.pyplot as plt
```

- OpenCV (cv2) : lecture, redimensionnement et conversion en niveaux de gris des images.
- os : navigation dans le système de fichiers pour charger le dataset.
- NumPy : manipulation efficace des tableaux d'images.
- tqdm : barre de progression pour le chargement des données.
- TFLearn : API haut niveau simplifiant la construction du CNN (couches conv_2d, max_pool_2d, fully_connected, dropout, regression).

#### Définir les constantes

In [ ]:
TRAIN_DIR = 'E:/dataset/Cats_vs_Dogs/train'
TEST_DIR = 'E:/dataset/Cats_vs_Dogs/test1'
IMG_SIZE = 50   # Redimensionner les images en 50x50 pixels
LR = 1e-3       # Taux d'apprentissage
MODEL_NAME = 'dogsvscats-{}-{}.model'.format(LR, '6conv-basic')

## Prétraiter les données

Chaque nom de fichier contient le label (ex : « cat.1.jpg » ou « dog.2.jpg »). On extrait ce label et on le convertit en encodage one-hot.

In [ ]:
def label_img(img):
    word_label = img.split('.')[-3]  # Extrait le label depuis le nom de fichier
    if word_label == 'cat': return [1, 0]  # One-hot pour "chat"
    elif word_label == 'dog': return [0, 1]  # One-hot pour "chien"

On convertit ensuite chaque image en niveaux de gris (pour réduire la complexité de calcul tout en préservant les formes et features), puis on la redimensionne en 50×50 pixels pour que toutes les images aient les mêmes dimensions.

In [ ]:
def create_train_data():
    training_data = []
    for img in tqdm(os.listdir(TRAIN_DIR)):
        label = label_img(img)
        path = os.path.join(TRAIN_DIR, img)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)  # Conversion en niveaux de gris
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))    # Redimensionnement
        training_data.append([np.array(img), np.array(label)])
    shuffle(training_data)  # Mélange pour éviter les biais
    np.save('train_data.npy', training_data)  # Sauvegarde pour réutilisation
    return training_data

## Construire et entraîner le modèle CNN

On construit le CNN avec TFLearn : une couche d'entrée, plusieurs couches convolution + pooling, des couches entièrement connectées, et une couche de sortie.

```python
tf.compat.v1.reset_default_graph()

convnet = input_data(shape=[None, IMG_SIZE, IMG_SIZE, 1], name='input')

convnet = conv_2d(convnet, 32, 5, activation='relu')
convnet = max_pool_2d(convnet, 5)

convnet = conv_2d(convnet, 64, 5, activation='relu')
convnet = max_pool_2d(convnet, 5)

convnet = conv_2d(convnet, 128, 5, activation='relu')
convnet = max_pool_2d(convnet, 5)

convnet = conv_2d(convnet, 64, 5, activation='relu')
convnet = max_pool_2d(convnet, 5)

convnet = conv_2d(convnet, 32, 5, activation='relu')
convnet = max_pool_2d(convnet, 5)

convnet = fully_connected(convnet, 1024, activation='relu')
convnet = dropout(convnet, 0.8)  # Dropout pour éviter l'overfitting

convnet = fully_connected(convnet, 2, activation='softmax')
convnet = regression(convnet, optimizer='adam', learning_rate=LR, loss='categorical_crossentropy', name='targets')

model = tflearn.DNN(convnet, tensorboard_dir='log')
```

#### Entraîner le modèle

In [ ]:
train_data = create_train_data()
train = train_data[:-500]  # Ensemble d'entraînement
test = train_data[-500:]   # Ensemble de validation

X = np.array([i[0] for i in train]).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
Y = np.array([i[1] for i in train])
test_x = np.array([i[0] for i in test]).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
test_y = np.array([i[1] for i in test])

model.fit({'input': X}, {'targets': Y}, n_epoch=5,
          validation_set=({'input': test_x}, {'targets': test_y}),
          snapshot_step=500, show_metric=True, run_id=MODEL_NAME)

model.save(MODEL_NAME)

## Tester et améliorer le modèle

On charge les données de test, on utilise le modèle entraîné pour prédire les labels, et on visualise les résultats :

In [ ]:
test_data = np.load('test_data.npy')
fig = plt.figure()

for num, data in enumerate(test_data[:20]):  # Test sur les 20 premières images
    img_data = data[0]
    y = fig.add_subplot(4, 5, num + 1)
    orig = img_data
    data_reshaped = img_data.reshape(IMG_SIZE, IMG_SIZE, 1)
    model_out = model.predict([data_reshaped])[0]
    str_label = 'Dog' if np.argmax(model_out) == 1 else 'Cat'
    y.imshow(orig, cmap='gray')
    plt.title(str_label)
    y.axes.get_xaxis().set_visible(False)
    y.axes.get_yaxis().set_visible(False)
plt.show()

#### Évaluer et améliorer

- Expérimente avec les hyperparamètres : taux d'apprentissage, nombre d'epochs, architecture du réseau.
- Utilise des techniques de data augmentation pour améliorer la performance du modèle.
- Surveille l'accuracy d'entraînement et de validation pour éviter l'overfitting.

> 💡 **Conclusion** : Tu as maintenant construit avec succès un classifieur d'images avec un CNN ! Ce modèle peut distinguer les chats des chiens avec une précision raisonnable. Tu peux l'améliorer encore en réglant les hyperparamètres, en utilisant un dataset plus large, ou en expérimentant des architectures plus avancées.

---
# Exercices de la journée
Ouvre chaque notebook d'exercices dans Colab, puis rends-le via GitHub sur Propilote-Learning :

- [Exercises XP : classification de fleurs avec un CNN](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-5/jour-3/exercices-xp-classification-fleurs.ipynb)
- [Exercises XP Gold](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-5/jour-3/exercices-xp-gold.ipynb)
- [Exercises XP Ninja](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-5/jour-3/exercices-xp-ninja.ipynb)
- [Daily Challenge : classifier des chiffres manuscrits avec des CNN](https://colab.research.google.com/github/franckkohou/propilote-notebooks/blob/main/semaine-5/jour-3/daily-challenge-chiffres-manuscrits-cnn.ipynb)